In [ ]:
import os
import pathlib
import logging
import shutil
import tarfile

import numina.util.context as ctx
from numina.user.session import Session
from numina.testing.testcache import download_cache
from megaradrp.tools.init_tree import ensure_numina_cfg, init_tree, install_calibration_data

import matplotlib.pyplot as plt

In [ ]:
logging.basicConfig(level=logging.DEBUG)

In [ ]:
workdir = pathlib.Path().resolve()

In [ ]:
dataset = 'megara-cookbook-v4'
url = 'https://guaix.fis.ucm.es/data/megaradrp/{}.tar.gz'.format(dataset)

downloaded = download_cache(url)

# Uncompress
with tarfile.open(downloaded.name, mode="r:gz") as tar:
    tar.extractall(filter="data")

os.remove(downloaded.name)

In [ ]:
# The M71 dataset does not have bias images, the master bias is reduced with the images of M15
basedir_m15 = workdir / dataset / 'M15_LCB_HR-R'
os.chdir(basedir_m15)
session_m15 = Session(basedir=basedir_m15)
session_m15.add_observations("0_Bias.yaml")

In [ ]:
obsid = "0_Bias"
task0 = session_m15.run(obsid)

In [ ]:
basedir = workdir / dataset / 'M71_MOS_LR-R'
# .numina.cfg and the calibration tree are in the directory of M71, as with megaradrp-inittree
os.chdir(basedir)
logger = logging.getLogger('inittree')
calibrations_url = 'https://zenodo.org/records/18623771/files/guaix-ucm/megaradrp-calibrations-2026.2.zip'
init_tree(base_path='calibrations', logger=logger)
install_calibration_data(url=calibrations_url, base_path='calibrations', overwrite=False, logger=logger)
ensure_numina_cfg(logger=logger)

In [ ]:
# The master bias of M15 in the calibration tree of M71
master_bias = basedir_m15 / task0.request_runinfo['results_dir'] / 'master_bias.fits'
shutil.copy(master_bias, basedir / 'calibrations/MEGARA/ca3558e3-e50d-4bbc-86bd-da50a0998a48/MasterBias/')

In [ ]:
# Registry of the reductions, the products are found there in later reductions
session = Session(basedir=basedir, db="numina-db.json")

In [ ]:
obsresults = ["1_TraceMap.yaml", "3_WaveCalib.yaml",
         "4_FiberFlat.yaml", "8_MosImage.yaml"]

session.add_observations(*obsresults)

In [ ]:
obsid = "1_TraceMap_LR-R"
task1 = session.run(obsid)

In [ ]:
obsid = "3_WaveCalib_LR-R"
task3 = session.run(obsid, requirements={"store_pdf_with_refined_fits": 0})

In [ ]:
obsid = "4_FiberFlat_LR-R"
task4 = session.run(obsid)

In [ ]:
obsid = "8_MosImage_LR-R_M71"
task8 = session.run(obsid)